# M10 — protótipo de treino (Colab Pro)

**O objetivo deste notebook não é um modelo bom. É um número:** o **custo por época**.

O plano de treino (`docs/plans/m10-treino-vastai.md`) estima a Fase 3 em $600–1.500 por
proporção ao run de M5 — cujo custo real **não está registrado em lugar nenhum do repositório**.
Esse é o risco R1, e é o número mais frágil de todo o plano. Um run de ~300 h aqui mede o custo
por época de verdade e permite reprecificar antes de comprometer centenas de dólares.

## O que este notebook NÃO faz

- não treina até convergir (são poucas épocas, de propósito)
- não decide a arquitetura — isso é o bake-off de T4
- não produz artefato publicável

## Configuração alvo

| | |
|---|---|
| GPU | **L4** (melhor custo/hora normalizado no Colab Pro) |
| Corpus | ~300 h do TAGARELA (~31 shards) |
| Features | fbank 80-dim em **lilcom_chunky** — 33 MB/h `[MEDIDO]` |
| Disco | ~10 GB de features + ~17 GB de parquets temporários |
| Checkpoints | no Drive, para sobreviver à sessão |


## 1. Ambiente

Colab Pro **não tem background execution** (isso é Pro+). A aba precisa ficar aberta.
Com checkpoint por época, uma desconexão vira atraso, não perda.


In [1]:
import subprocess, sys, os, json, time
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| disponivel', torch.cuda.is_available())
print('disco livre:', subprocess.run(['df','-h','/content'], capture_output=True, text=True).stdout.splitlines()[-1])


NVIDIA L4, 23034 MiB
torch 2.11.0+cu128 | cuda 12.8 | disponivel True
disco livre: overlay         236G   48G  189G  21% /


## 2. Dependências

⚠️ **O ponto mais frágil do notebook é o `k2`**: a wheel precisa casar exatamente com a versão
de torch e CUDA do runtime. Se a célula falhar, confira a matriz em
<https://k2-fsa.github.io/k2/installation/pre-compiled-cuda-wheels-linux/> e ajuste a URL.

O `k2stub` do repositório **não serve aqui** — ele cobre só a ativação Swoosh para smoke em
CPU, e a receita de treino usa `k2.ctc_loss` e grafos.


In [2]:
# --- k2: escolher a wheel EXATA, nunca deixar o pip resolver ---
# O PyPI tem um pacote chamado "k2" que NAO e este: ele instala um __init__.py
# sem a extensao compilada _k2, e o erro so aparece no import.
# Por isso aqui nao existe "pip install k2": a wheel e escolhida pelo indice
# oficial e instalada por URL exata.
import re, urllib.request, subprocess, sys

INDEX   = 'https://k2-fsa.github.io/k2/cuda.html'
PY_TAG  = f'cp{sys.version_info.major}{sys.version_info.minor}'
TORCH   = torch.__version__.split('+')[0]
CUDA    = torch.version.cuda or ''
print(f'runtime: python {PY_TAG}  torch {TORCH}  cuda {CUDA}')

html  = urllib.request.urlopen(INDEX, timeout=120).read().decode('utf-8', 'replace')
urls  = sorted(set(re.findall(r'https://[^"\s]+\.whl', html)))

def campos(u):
    m = re.search(r'k2-([\d.]+(?:\.dev\d+)?)\+cuda([\d.]+)\.torch([\d.]+)-(cp\d+)-', u)
    return m.groups() if m else None   # (versao, cuda, torch, pytag)

cand = [(c, u) for u in urls if (c := campos(u))
        and c[3] == PY_TAG and c[2] == TORCH]
if not cand:
    raise RuntimeError(
        f'Nenhuma wheel k2 para {PY_TAG} + torch {TORCH}.\n'
        f'Torch disponiveis para {PY_TAG}: '
        + ', '.join(sorted({c[2] for u in urls if (c := campos(u)) and c[3] == PY_TAG}))
        + '\nOpcao: fixar o torch do runtime numa versao coberta.')

exato = [x for x in cand if x[0][1] == CUDA]
if exato:
    escolha = exato
else:
    # Subir de CUDA minor e a direcao perigosa: uma wheel de 12.9 pode exigir
    # driver mais novo do que o runtime de 12.4 oferece. Preferir o maior minor
    # que seja <= o do runtime, e so subir se nao houver nenhum abaixo.
    def minor(v):
        p = v.split('.')
        return (int(p[0]), int(p[1]) if len(p) > 1 else 0)
    alvo   = minor(CUDA)
    mesmos = [x for x in cand if minor(x[0][1])[0] == alvo[0]]
    if not mesmos:
        raise RuntimeError(f'k2 existe para torch {TORCH}/{PY_TAG}, mas nao para CUDA {alvo[0]}.x. '
                           f'CUDAs: {sorted({c[1] for c, _ in cand})}')
    abaixo  = [x for x in mesmos if minor(x[0][1]) <= alvo]
    escolha = abaixo or mesmos
    usado   = max(minor(x[0][1]) for x in escolha)
    escolha = [x for x in escolha if minor(x[0][1]) == usado]
    print(f'AVISO: sem wheel para CUDA {CUDA} exata; usando {usado[0]}.{usado[1]}'
          + ('' if abaixo else ' (ACIMA do runtime -- pode exigir driver mais novo)')
          + '. Compatibilidade de minor version costuma funcionar, mas NAO esta verificada aqui.')

(ver, cu, tv, _), url = sorted(escolha, key=lambda x: x[0][0])[-1]
print(f'instalando k2 {ver} (cuda {cu}, torch {tv}) -- ~177 MB')

# --no-deps: a wheel declara torch como dependencia e o pip trocaria o torch do
# runtime, o que quebra a CUDA da sessao.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', url], check=True)
# lilcom NAO vem com o lhotse: `from lhotse.features.io import LilcomChunkyWriter`
# importa sem erro e a falha so aparece quando um worker abre o storage, depois de
# todo o filtro de corpus ter rodado. Ver requirements-train.txt.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lhotse', 'lilcom', 'kaldialign', 'sentencepiece', 'kaldi_native_fbank',
                'num2words', 'regex'], check=True)
import lilcom  # falha aqui, nao dentro de um worker 20 min depois
print('lilcom', lilcom.__version__ if hasattr(lilcom, '__version__') else 'ok')

import importlib.metadata as md
import k2

# A versao vem do dist-info, nao de um atributo do modulo: k2 nem sempre expoe
# __version__, e a string do dist-info carrega a tag local (+cuda12.8.torch2.11.0),
# que e justamente a prova de QUAL wheel ficou instalada.
instalado = md.version('k2')
print('k2', instalado)
assert f'cuda{cu}' in instalado and f'torch{tv}' in instalado, (
    f'wheel instalada ({instalado}) nao e a escolhida (cuda {cu}, torch {tv}) -- '
    'provavelmente sobrou o pacote errado do PyPI; Runtime > Restart session e rode de novo')

# Prova funcional: um kernel do k2 rodando na GPU. E um teste, nao uma suposicao
# sobre o nome de um atributo -- se a API mudou, o notebook mostra o que existe
# em vez de morrer em AttributeError.
try:
    _r = k2.RaggedTensor([[1, 2], [3]]).to(torch.device('cuda'))
    print('k2 com CUDA: ok (RaggedTensor em', _r.device, ')')
except Exception as e:
    print('AVISO: prova funcional de CUDA nao rodou:', type(e).__name__, e)
    print('atributos publicos de k2:', [a for a in dir(k2) if not a.startswith('_')][:40])
    print('A versao instalada esta correta; siga, mas o primeiro passo do treino'
          ' e quem vai confirmar que os kernels funcionam.')


runtime: python cp313  torch 2.11.0  cuda 12.8
instalando k2 1.24.4.dev20260626 (cuda 12.8, torch 2.11.0) -- ~177 MB
lilcom ok
k2 1.24.4.dev20260626+cuda12.8.torch2.11.0
k2 com CUDA: ok (RaggedTensor em cuda:0 )


## 3. Código: icefall + jvscribe

O `jvscribe` traz os preparadores de corpus já testados (`prep_tagarela.py`), com filtro
determinístico de alucinação e relatório de cobertura por show.


In [3]:
%cd /content

# Ambos publicos: clone direto, sem token.
!test -d icefall  || git clone -q https://github.com/k2-fsa/icefall.git
# --branch workspace NAO e detalhe: a default do repo e main, que esta 125 commits
# atras e AINDA GRAVA FEATURES EM NUMPY (115 MB/h contra 33 MB/h do lilcom).
!test -d jvscribe || git clone -q --branch workspace https://github.com/paulohenriquevn/jvscribe.git

PREP = '/content/jvscribe/jvscribe/finetune/prep_tagarela.py'
assert os.path.isdir('/content/icefall'), 'clone do icefall falhou'
assert os.path.exists(PREP), (
    'clone do jvscribe falhou -- na versao anterior desta celula isso passava em '
    'silencio e so aparecia tres celulas depois')

# Confere o CONTEUDO, nao so a presenca: um clone da branch errada tem o arquivo
# e grava numpy. Falhar aqui custa um segundo; descobrir depois do fbank custa
# a extracao inteira.
assert 'LilcomChunkyWriter' in open(PREP).read(), (
    'prep_tagarela.py sem LilcomChunkyWriter -- branch errada. '
    'Use --branch workspace.')

os.environ['PYTHONPATH'] = '/content/icefall:' + os.environ.get('PYTHONPATH', '')
!pip install -q -r /content/icefall/requirements.txt 2>&1 | tail -3
print('icefall + jvscribe (workspace) prontos')


/content
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
icefall + jvscribe (workspace) prontos


## 4. Checkpoints no Drive

A sessão morre; o Drive não. **Só os checkpoints vão para o Drive** — as features ficam no
disco local, que é ordens de magnitude mais rápido para o dataloader.


In [4]:
from google.colab import drive
drive.mount('/content/drive')
EXP = '/content/drive/MyDrive/jvscribe/m10_proto/exp'
os.makedirs(EXP, exist_ok=True)
print('checkpoints ->', EXP)


Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/jvscribe/m10_proto/exp


## 5. Corpus

Amostragem **estratificada**: shards espaçados uniformemente ao longo dos 1.764. Como
**um shard é um show** (`wiki/medicoes/m10-t3-quanto-corpus-pt-existe.md`), espaçar por
posição é espaçar por show.

Cada shard tem ~9,62 h `[MEDIDO]`, então 31 shards ≈ 300 h.


In [5]:
N_SHARDS = 31           # ~300 h; cada shard ~9,62 h [MEDIDO]
TOTAL_SHARDS = 1764
RAW = '/content/tagarela_raw'; os.makedirs(RAW, exist_ok=True)
from huggingface_hub import hf_hub_download
idx = [round(i*(TOTAL_SHARDS-1)/(N_SHARDS-1)) for i in range(N_SHARDS)]
t0 = time.time()
for k, i in enumerate(idx):
    f = f'data/train-{i:05d}-of-01764.parquet'
    hf_hub_download('freds0/TAGARELA', f, repo_type='dataset', local_dir=RAW)
    if (k+1) % 5 == 0: print(f'  {k+1}/{N_SHARDS}  {time.time()-t0:.0f}s', flush=True)
print(f'{N_SHARDS} shards em {(time.time()-t0)/60:.1f} min')


data/train-00000-of-01764.parquet: reconstructing file:   0%|          |  0.00B /  852MB            

data/train-00000-of-01764.parquet: downloading bytes:           |  0.00B            

data/train-00059-of-01764.parquet: reconstructing file:   0%|          |  0.00B /  838MB            

data/train-00059-of-01764.parquet: downloading bytes:           |  0.00B            

data/train-00118-of-01764.parquet: reconstructing file:   0%|          |  0.00B /  842MB            

...

data/train-01704-of-01764.parquet: downloading bytes:           |  0.00B            

  30/31  139s


data/train-01763-of-01764.parquet: reconstructing file:   0%|          |  0.00B /  556MB            

data/train-01763-of-01764.parquet: downloading bytes:           |  0.00B            

31 shards em 2.4 min


### 5.1 Cuts + fbank

`prep_tagarela.py` filtra alucinação, gera o relatório de cobertura por show e grava features
em **lilcom_chunky** — 33 MB/h contra 115 MB/h do default numpy `[MEDIDO]`.


In [6]:
!cd /content/jvscribe && python3 jvscribe/finetune/prep_tagarela.py \
    --parquet-dir /content/tagarela_raw/data \
    --out /content/data/tagarela \
    --num-jobs 4
!du -sh /content/data/tagarela/feats_train
!cat /content/data/tagarela/show_distribution.txt 2>/dev/null | head -5


[tagarela] total=124970 kept=113311 empty_text=26 wrong_accent=10992 hallucination=549 bad_ratio=92 decode_error=0
[tagarela] cobertura: 187 shows/episódios distintos em 113311 utts mantidas
[tagarela]      7887  show_0VoRHJEAvyuFI2TJItfwru
[tagarela]      7527  show_1aQ5cqZ8ST0G7oYPcUYBW4
[tagarela]      3939  show_1kLfOhEgureVSOpRUGGgih
[tagarela]      3903  show_1dnK97Wql8o6SVwxzSQ7bb
[tagarela]      3874  show_1oQJjn3RA8MO7dfeMF0ExR
[tagarela]      3865  show_0t2YcQLuHgPvCeo6AnGpos
[tagarela]      3859  show_1xbd6Gfk8xiL6YpYdpQL3k
[tagarela]      3848  show_0o2D2ABgW9ad6cgcZrj89L
[tagarela]      3770  show_0PugtmJttCW1q8sGid0vnV
[tagarela]      3546  show_00uu4KLVIPI0C5JXLQGncS
[tagarela]      3495  show_12DZPbJ8CI0o8yavD3OgzZ
[tagarela]      3356  show_13zR6wiTJW4w1T5srY3lHb
[tagarela]      3346  show_1Kspfitqtgme8SDA8OkxWU
[tagarela]      3156  show_1FNEP5Y4CiPAOVXa2pbfWQ
[tagarela]      3116  show_1gFLOGRvqRKEV9SU8rbgyz
[tagarela] maior show = 7.0% das utts
Extracting and storin

### 5.2 Verificar o storage — não confie no parâmetro

O lhotse aceitou `storage_type` e gravou numpy assim mesmo numa execução real
(`wiki/medicoes/m10-t3-fbank-e-storage.md`). **Conferir custa um segundo; errar custa 410 GB**
na escala de 5.000 h.


In [7]:
import gzip, json as _json
with gzip.open('/content/data/tagarela/tagarela_cuts_train.jsonl.gz','rt') as f:
    c = _json.loads(f.readline())
st = c['features']['storage_type']
print('storage_type =', st)
assert st == 'lilcom_chunky', f'ESPERADO lilcom_chunky, veio {st} -- 3,5x mais disco'


storage_type = lilcom_chunky


### 5.3 Nomear os manifests como o datamodule espera

O datamodule do CommonVoice carrega `{cv_manifest_dir}/cv-{language}_cuts_{split}.jsonl.gz`
(`asr_datamodule.py:409` e `:432`). Nossos cuts se chamam `tagarela_cuts_train` — e o próprio
`prep_tagarela.py` avisa, na docstring, que esse prefixo **não casa o glob default**.

Sem este passo o treino procura `data/en/fbank/cv-en_cuts_train.jsonl.gz` e morre.


In [8]:
from lhotse import CutSet
CV, LANGID, N_DEV = '/content/data/tagarela', 'pt', 1000

cuts = CutSet.from_file(f'{CV}/tagarela_cuts_train.jsonl.gz')
ids  = list(cuts.ids)
assert len(ids) > N_DEV * 5, f'so {len(ids)} cuts -- dev de {N_DEV} come demais'

# Fatia CONTIGUA do fim. Os cut ids sao um contador na ordem em que os shards foram
# lidos, e um shard e um show, entao o fim tende a ser 1-2 shows.
# HONESTIDADE: isto NAO e disjuncao de show provada -- prep_tagarela.py conta o show
# para o relatorio mas NAO grava no cut, entao daqui nao da para verificar. Para medir
# CUSTO por epoca, que e o objetivo deste notebook, nao importa. Para comparar WER
# entre receitas, importa, e o campo precisa existir antes.
dev_ids = set(ids[-N_DEV:])
dev   = cuts.subset(cut_ids=list(dev_ids))
train = cuts.filter(lambda c: c.id not in dev_ids)

dev.to_file(f'{CV}/cv-{LANGID}_cuts_dev.jsonl.gz')
train.to_file(f'{CV}/cv-{LANGID}_cuts_train.jsonl.gz')

HORAS_TREINO = sum(c.duration for c in CutSet.from_file(f'{CV}/cv-{LANGID}_cuts_train.jsonl.gz'))/3600
HORAS_DEV    = sum(c.duration for c in dev)/3600
print(f'train {len(ids)-N_DEV} cuts / {HORAS_TREINO:.1f} h')
print(f'dev   {N_DEV} cuts / {HORAS_DEV:.1f} h')


train 112311 cuts / 294.6 h
dev   1000 cuts / 2.1 h


## 6. Tokenizer BPE

⚠️ O `bpe.model` do M4 **se perdeu** e custou retrabalho. Aqui ele vai para o Drive junto dos
checkpoints, porque sem ele o checkpoint é inútil.


In [9]:
LANG = '/content/data/lang_bpe_500'; os.makedirs(LANG, exist_ok=True)
import sentencepiece as spm

# user_defined_symbols NAO e opcional: train.py faz
#   params.blank_id = sp.piece_to_id("<blk>")            (train.py:1138)
# e, sem o simbolo, piece_to_id devolve o id do <unk>. Foi o que aconteceu na
# primeira tentativa -- o log trouxe "blank_id": 2, ou seja, blank COLIDINDO com unk.
# Nada falha; o modelo so treina errado. icefall define os dois simbolos primeiro,
# entao <blk>=0, <sos/eos>=1, <unk>=2 (local/train_bpe_model.py:85).
# pad_id foi removido: valia 0 e colidia com o <blk>.
spm.SentencePieceTrainer.train(
    input='/content/data/tagarela/transcript_words.txt',
    model_prefix=f'{LANG}/bpe', vocab_size=500, model_type='bpe',
    character_coverage=1.0,
    user_defined_symbols=['<blk>', '<sos/eos>'],
    unk_id=2, bos_id=-1, eos_id=-1)

_sp = spm.SentencePieceProcessor(model_file=f'{LANG}/bpe.model')
assert _sp.piece_to_id('<blk>') == 0, f"<blk> ficou em {_sp.piece_to_id('<blk>')}, deveria ser 0"
print('vocab', _sp.get_piece_size(), '| <blk>=0 ok')

!cp {LANG}/bpe.model {EXP}/   # sobrevive a sessao
print('bpe.model ->', EXP)


vocab 500 | <blk>=0 ok
bpe.model -> /content/drive/MyDrive/jvscribe/m10_proto/exp


## 7. Treino instrumentado

As armadilhas já pagas entram como configuração, não como descoberta. As seis últimas
foram cobradas por este notebook, nas três primeiras tentativas de treino:

| armadilha | configuração |
|---|---|
| fp16 colapsa sob augmentação | `--use-fp16 0` |
| as 4 flags não estão no `.pt` | passadas explicitamente |
| disco cheio trunca o checkpoint **sem erro** | contagem + tamanho dos `.pt` |
| manifest com prefixo próprio é ignorado | `--cv-manifest-dir` + `--language` |
| MUSAN não existe aqui | `--enable-musan 0` |
| `<blk>` ausente vira id do `<unk>` | `user_defined_symbols` no BPE |
| **LR alto + batch pequeno → divergência** | `--max-duration 700` + `--base-lr 0.015` |

A última é a mais cara e a mais fácil de repetir: `--base-lr 0.03` veio da lição de M5,
que era um **finetune de cabeça fresca sobre encoder pré-treinado**. Aqui o treino é **do
zero**, e a razão que importa é `lr / batch efetivo`. O `RESULTS.md` do recipe treina a
2200 s de batch com lr 0.045 — razão 2.0e-5. Com 300 s e lr 0.03 a razão vira 1.0e-4, cinco
vezes mais quente, e a loss sobe até NaN.

**A instrumentação é o produto deste notebook**: horas por época, e o custo em unidades.
Custo por época mede **throughput**, não convergência — o forward e o backward custam o
mesmo numa época que aprende e numa que diverge. Por isso a célula reporta o custo desde
que **uma época tenha completado**, e diz em seguida se o modelo presta.


In [10]:
UNIDADES_POR_HORA = {'L4': 4.8, 'A100': 13.0, 'T4': 2.0}   # [ESTIMATIVA] nao medido
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
un_h = next((v for k,v in UNIDADES_POR_HORA.items() if k in GPU), None)
print(f'GPU: {GPU} | unidades/h: {un_h}')

EPOCAS = 3        # suficiente para medir custo/epoca; NAO para convergir

# MAX_DUR e BASE_LR andam JUNTOS -- foi a combinacao dos dois que divergiu.
# A primeira tentativa usou 300 s com lr 0.03 e a loss subiu ate NaN no batch
# ~3300. O RESULTS.md do proprio recipe treina com batch efetivo de ~2200 s
# (world-size 2 x max-duration 1000, ou 4 x 550) e base-lr 0.045 -- razao
# lr/batch de 2.0e-5. A nossa era 1.0e-4: cinco vezes mais quente.
#   300 s foi escolhido por medo de OOM; o log mostrou pico de 9.090 MB numa
#   L4 de 24 GB, entao havia folga de 2,5x que nao estava sendo usada.
MAX_DUR = 700     # ~9 GB medidos em 300 s; sobe proporcional. Baixar se der OOM.
BASE_LR = 0.015   # 0.015/700 = 2.1e-5, a razao do recipe

# --cv-manifest-dir + --language: sem eles o datamodule procura
#   data/en/fbank/cv-en_cuts_train.jsonl.gz     (o default da receita CommonVoice)
# --enable-musan 0: com musan ligado ele carrega {manifest_dir}/musan_cuts.jsonl.gz
#   (asr_datamodule.py:231), que nao existe aqui.
# --use-transducer 1: o alvo do M10 e RNN-T streaming. A perda podada do transducer
#   e justamente a parte cara, entao medir so CTC produziria um custo que nao
#   transfere para a receita real.
cmd = (
    'cd /content/icefall/egs/commonvoice/ASR && python3 zipformer/train.py'
    f' --world-size 1 --num-epochs {EPOCAS} --start-epoch 1'
    f' --exp-dir {EXP} --bpe-model {LANG}/bpe.model'
    f' --cv-manifest-dir {CV} --language {LANGID}'
    ' --enable-musan 0'
    f' --max-duration {MAX_DUR} --use-fp16 0 --num-workers 2'
    ' --num-encoder-layers 2,2,3,4,3,2'
    ' --feedforward-dim 512,768,1024,1536,1024,768'
    ' --encoder-dim 192,256,384,512,384,256'
    ' --encoder-unmasked-dim 192,192,256,256,256,192'
    f' --causal 1 --use-transducer 1 --use-ctc 1 --base-lr {BASE_LR}'
)
print(cmd)


GPU: NVIDIA L4 | unidades/h: 4.8
cd /content/icefall/egs/commonvoice/ASR && python3 zipformer/train.py --world-size 1 --num-epochs 3 --start-epoch 1 --exp-dir /content/drive/MyDrive/jvscribe/m10_proto/exp --bpe-model /content/data/lang_bpe_500/bpe.model --cv-manifest-dir /content/data/tagarela --language pt --enable-musan 0 --max-duration 700 --use-fp16 0 --num-workers 2 --num-encoder-layers 2,2,3,4,3,2 --feedforward-dim 512,768,1024,1536,1024,768 --encoder-dim 192,256,384,512,384,256 --encoder-unmasked-dim 192,192,256,256,256,192 --causal 1 --use-transducer 1 --use-ctc 1 --base-lr 0.015


In [11]:
# `!{cmd}` engole o exit code: na primeira tentativa o treino morreu em 40 s e esta
# celula imprimiu "$0.00 por epoca" como se fosse medicao. Um numero sobre um run
# que falhou nao mede nada -- entao o exit code manda.
#
# E o log vai para arquivo: na segunda tentativa o treino rodou 77 min e a excecao
# nao dizia por que. 77 min de saida nao cabem no output da celula.
#
# A terceira tentativa divergiu: a loss SUBIU por 3.300 batches ate virar NaN, e
# so entao o icefall abortou -- 75 min gastos depois que o problema ja era visivel.
# Por isso ha um vigia: a tot_loss de cada log e comparada com a menor ja vista, e
# o treino e morto se subir 25% e ficar assim. Custo/epoca ja estara medido.
import re, signal

LOG = '/content/train.log'
print(f'log -> {LOG}  (tail -f em outra celula para acompanhar)')

RE_TOT = re.compile(r'tot_loss\[loss=([\d.]+)')
melhor, ruins, MAX_RUINS, FATOR = float('inf'), 0, 4, 1.25

t0 = time.time()
with open(LOG, 'w') as fh:
    proc = subprocess.Popen(['bash', '-lc', cmd], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1,
                            start_new_session=True)
    divergiu = False
    for linha in proc.stdout:
        fh.write(linha); fh.flush()
        if any(k in linha for k in ('Epoch', 'Error', 'error', 'Traceback',
                                    'CUDA', 'Saving', 'validation')):
            print(linha, end='')
        m = RE_TOT.search(linha)
        if m:
            v = float(m.group(1))
            if v != v or v > melhor * FATOR:      # NaN ou subiu demais
                ruins += 1
                if ruins >= MAX_RUINS:
                    divergiu = True
                    print(f'\n!! DIVERGIU: tot_loss {v:.3f} contra minimo {melhor:.3f} '
                          f'em {MAX_RUINS} logs seguidos. Matando o treino.')
                    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
                    break
            else:
                ruins = 0
                melhor = min(melhor, v)
    proc.wait()
el = time.time() - t0

# O custo por epoca e uma medida de THROUGHPUT, nao de qualidade: o forward e o
# backward custam o mesmo numa epoca que converge e numa que diverge. Entao o
# numero vale mesmo aqui -- desde que o run tenha durado pelo menos uma epoca.
ckpts = sorted(f for f in os.listdir(EXP) if f.startswith('epoch-') and f.endswith('.pt'))
for f in ckpts:
    print(f'  {f}  {os.path.getsize(os.path.join(EXP, f))/1e6:.0f} MB')

if not ckpts:
    cauda = open(LOG).read().splitlines()[-40:]
    print('\n'.join(cauda))
    raise RuntimeError(f'nenhuma epoca completou em {el/60:.1f} min -- sem custo a medir. '
                       f'Log em {LOG}.')

h_por_epoca = el/3600/len(ckpts)
print(f'\n=== CUSTO MEDIDO ===   ({HORAS_TREINO:.0f} h de corpus, {len(ckpts)} epoca(s))')
print(f'{h_por_epoca:.2f} h por epoca')
if un_h:
    print(f'{h_por_epoca*un_h:.1f} unidades = ${h_por_epoca*un_h/10:.2f} por epoca')
    for horas_corpus in (1500, 5000):
        fator = horas_corpus/HORAS_TREINO
        print(f'  extrapolado p/ {horas_corpus} h x 10 epocas: '
              f'${h_por_epoca*un_h/10*fator*10:.0f}  [ESTIMATIVA linear]')

if divergiu or proc.returncode != 0:
    print(f'\nO CUSTO ACIMA VALE. O MODELO NAO: exit {proc.returncode}, divergiu={divergiu}.')


log -> /content/train.log  (tail -f em outra celula para acompanhar)
2026-09-21 15:33:20.561588: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-21 15:34:33,358 INFO [train.py:1054] Epoch 1, batch 0, loss[loss=8.99, simple_loss=7.367, pruned_loss=6.89, ctc_loss=4.662, over 17159.00 frames. ], tot_loss[loss=8.99, simple_loss=7.367, pruned_loss=6.89, ctc_loss=4.662, over 17159.00 frames. ], batch size: 38, lr: 7.51e-03, 
2026-09-21 15:34:33,360 INFO [train.py:1077] Computing validation loss
2026-09-21 15:34:41,789 INFO [train.py:1086] Epoch 1, validation: loss=8.882, simple_loss=7.257, pruned_loss=6.736, ctc_loss=4.752, over 189937.00 frames. 
2026-09-21 15:36:14,131 INFO [train.py:1054] Epoch 1, batch 50, loss[loss=3.126, simple_loss=2.586, pruned_loss=1.46, ctc_lo

### 7.1 Se falhou: onde o treino parou

O icefall grava seu próprio log em `{exp-dir}/log/`, e o exp-dir está no **Drive** — então
ele sobrevive à sessão, mesmo quando o output da célula se perde.


In [12]:
# Forense do run que falhou. O exp-dir esta no Drive, entao SOBREVIVE a sessao:
# o icefall grava log/log-train-*.txt la, e e ele que diz onde o treino parou.
import glob, os

print('=== o que sobrou no exp-dir (Drive) ===')
for f in sorted(glob.glob(f'{EXP}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f'  {os.path.getsize(f)/1e6:9.1f} MB  {f.replace(EXP+"/", "")}')

logs = sorted(glob.glob(f'{EXP}/log/log-train-*.txt'), key=os.path.getmtime)
if not logs:
    print('\nsem log/log-train-*.txt -- o treino morreu antes de abrir o log')
else:
    print(f'\n=== ultimas 40 linhas de {os.path.basename(logs[-1])} ===')
    print('\n'.join(open(logs[-1], errors="replace").read().splitlines()[-40:]))

print('\n=== disco da sessao ===')
!df -h /content | tail -1
!du -sh /content/data/tagarela/* 2>/dev/null | sort -h | tail -5


=== o que sobrou no exp-dir (Drive) ===
     1063.0 MB  bad-model-0.pt
        9.5 MB  batch-c33f4584-b23b-c1d8-493c-d01609de8895.pt
     1063.0 MB  best-train-loss.pt
     1063.0 MB  best-valid-loss.pt
        0.2 MB  bpe.model
     1063.0 MB  checkpoint-4000.pt
     1063.0 MB  epoch-1.pt
     1063.0 MB  epoch-2.pt
     1063.0 MB  epoch-3.pt
        0.0 MB  log/log-train-2026-09-21-03-31-22
        1.3 MB  log/log-train-2026-09-21-12-53-36
        1.6 MB  log/log-train-2026-09-21-15-33-28
        0.0 MB  tensorboard/events.out.tfevents.1789961482.ada4bb65fa62.24258.0
        0.1 MB  tensorboard/events.out.tfevents.1789995216.b6f1c1a09ca1.30724.0
        0.1 MB  tensorboard/events.out.tfevents.1790004809.d630557970c8.21304.0

sem log/log-train-*.txt -- o treino morreu antes de abrir o log

=== disco da sessao ===
overlay         236G  116G  121G  49% /
14M	/content/data/tagarela/cv-pt_cuts_train.jsonl.gz
15M	/content/data/tagarela/tagarela_cuts_train.jsonl.gz
16M	/content/data/tagarela

## 8. O que fazer com o número

1. Registrar em `wiki/medicoes/` com a GPU, o corpus e as épocas ao lado — sem isso o número
   não transfere.
2. Reprecificar a Fase 3 de `docs/plans/m10-treino-vastai.md`, substituindo a proporção
   herdada de M5 pela medição.
3. **A extrapolação linear é otimista**: batch maior aproveita melhor a GPU, e corpus maior
   muda o gargalo de compute para I/O. Tratar como piso, não como estimativa.

### Se der OOM

Baixar `--max-duration` antes de qualquer outra coisa. O registro de M5 tem OOM em pre-scan
resolvido com `eager` + `workers=2`.

### Se a loss platôar em blank e o WER for 100%

É a armadilha do LR de cabeça fresca. Conferir que `--base-lr` está em 0.03 e não em 0.0001.
